In [4]:
# %%
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from geneticengine.grammar.decorators import abstract

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold

# %%
import warnings
warnings.filterwarnings('ignore')

# %%
for attemp in range(3):
    try:
        if os.path.exists('Base.csv'):
            df = pd.read_csv('Base.csv')
            print("Dataset loaded successfully")
            break
        else:
            import kagglehub
            import shutil
            path = kagglehub.dataset_download("sgpjesus/bank-account-fraud-dataset-neurips-2022")
            csv_path = os.path.join(path, "Base.csv")
            shutil.copy(csv_path, "Base.csv")
            df = pd.read_csv('Base.csv')
            print("Dataset downloaded and loaded successfully")
            break
    except Exception as e:
        print(f"Attempting again to download dataset due to error: {e}")
        if attemp < 2:
            time.sleep(5)
        else:
            raise e

# %%
df.info()

# %%
df.head(5)

# %%
train_val_df = df[df['month'] <= 5].sample(frac=1, random_state=42)
test_df = df[df['month'] >=6].sample(frac=1, random_state=42)

train_val_df.drop('month', axis=1, inplace=True)
test_df.drop('month', axis=1, inplace=True)

X_train_val = train_val_df.drop('fraud_bool', axis=1)
y_train_val = train_val_df['fraud_bool']
X_test = test_df.drop('fraud_bool', axis=1)
y_test = test_df['fraud_bool']


print(y_train_val.value_counts(), y_test.value_counts())

# %%
categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]


encoders = {}
for feat in categorical_features:
    encoder = LabelEncoder()
    X_train_val[feat] = encoder.fit_transform(X_train_val[feat])
    X_test[feat] = encoder.transform(X_test[feat])
    encoders[feat] = encoder

categorical_indices = [X_train_val.columns.get_loc(feat) for feat in categorical_features]

VALID_CONDITIONS = []
for abs_idx in categorical_indices:
    feat_name = X_train_val.columns[abs_idx]
    unique_values = np.unique(X_train_val.iloc[:, abs_idx])

    for val in unique_values:
        VALID_CONDITIONS.append((abs_idx, val, feat_name))

NUM_VALID_CONDITIONS = len(VALID_CONDITIONS)

n_features = X_train_val.shape[1]
all_indices = set(range(n_features))
cat_indices_set = set(categorical_indices)

# Create strictly numerical indices list
NUMERICAL_INDICES = sorted(list(all_indices - cat_indices_set))
NUM_NUMERICAL = len(NUMERICAL_INDICES)

print(f"Total features: {n_features}")
print(f"Numerical features available for math: {NUM_NUMERICAL}")
print(f"Categorical features available for conditions: {len(categorical_indices)}")

# %%
feature_names = X_train_val.columns.tolist()
n_features = len(feature_names)

# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# fpr_results = []
# for train_index, val_index in skf.split(X_train_val, y_train_val):
#     X_train, X_val = X_train_val.iloc[train_index], X_train_val.iloc[val_index]
#     y_train, y_val = y_train_val.iloc[train_index], y_train_val.iloc[val_index]
    
#     # model_baseline = lgb.LGBMClassifier(max_bins=63,device_type='gpu',n_estimators=350, max_depth=14, learning_rate=0.03, num_leaves=17, boosting_type='gbdt', random_state=42, verbose=-1, scale_pos_weight= (y_train==0).sum() / (y_train==1).sum())
#     model_baseline = lgb.LGBMClassifier(n_estimators=350, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train==0).sum() / (y_train==1).sum())
#     model_baseline.fit(X_train, y_train, categorical_feature=categorical_indices)

#     predictions = model_baseline.predict_proba(X_val)[:,1]
    
#     fprs, tprs, thresholds = roc_curve(y_val, predictions)
#     threshold = np.min(thresholds[fprs==max(fprs[fprs < 0.05])])
#     recall = np.max(tprs[fprs==max(fprs[fprs < 0.05])])
#     print(recall)
#     fpr_results.append(recall)

# baseline_tpr = np.mean(fpr_results)
# print("Average Recall at FPR < 5%:", baseline_tpr)

Dataset loaded successfully
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_2

In [8]:
@dataclass
class Value(ABC):
    def evaluate(self): pass

class Main(ABC):
    pass

@abstract
@dataclass
class Scalar(Main):
    """Base for strictly numerical values (Raw variables OR Operations)"""
    pass

@abstract
@dataclass
@weight(0.55)
class Operation(Scalar):
    """Subset of Scalar for computed values"""
    pass

@abstract
@dataclass
class Condition(ABC):
    def evaluate(self, X_np): pass

# --- TERMINALS ---

@weight(0.45)
@dataclass
class NumericalVar(Scalar): 
    # CHANGE 1: Use strictly numerical indices
    index: Annotated[int, IntRange(0, NUM_NUMERICAL - 1)]

    def evaluate(self, X_np):
        return X_np[:, NUMERICAL_INDICES[self.index]]
    
    def __str__(self):
        return feature_names[NUMERICAL_INDICES[self.index]]

@dataclass
class CategoricalCondition(Condition):
    # This is already correct; it uses your pre-calculated valid conditions
    index: Annotated[int, IntRange(0, NUM_VALID_CONDITIONS-1)]

    def evaluate(self, X_np):
        abs_idx, val, _ = VALID_CONDITIONS[self.index]
        return X_np[:, abs_idx] == val
    
    def __str__(self):
        _, val, feat_name = VALID_CONDITIONS[self.index]
        return f"({feat_name} == {val})"

# --- OPERATIONS (inherit from Operation) ---
# These take 'Scalar' as input, so they can take NumericalVar OR other Operations.
# They CANNOT take Categorical features because we didn't make a CategoricalVar that inherits from Scalar.

@weight(0.05)
@dataclass 
class Add(Operation):
    right: Scalar
    left: Scalar
    def evaluate(self, X_np): return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    def __str__(self): return f"({self.left} + {self.right})"

@weight(0.05)
@dataclass
class Subtract(Operation):
    right: Scalar
    left: Scalar
    def evaluate(self, X_np): return self.left.evaluate(X_np) - self.right.evaluate(X_np)
    def __str__(self): return f"({self.left} - {self.right})"

@weight(0.05)
@dataclass
class Multiply(Operation):
    right: Scalar
    left: Scalar
    def evaluate(self, X_np): return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    def __str__(self): return f"({self.left} * {self.right})"

@weight(0.05)
@dataclass
class Divide(Operation):
    right: Scalar
    left: Scalar
    def evaluate(self, X_np):
        denom = self.right.evaluate(X_np)
        return self.left.evaluate(X_np) / np.where(denom == 0, 1e-6, denom)
    def __str__(self): return f"({self.left} / {self.right})"
    
@weight(0.05)
@dataclass
class Sqrt(Operation):
    value: Scalar
    def evaluate(self, X_np): return np.sqrt(np.clip(self.value.evaluate(X_np), 0, None))
    def __str__(self): return f"sqrt({self.value})"
    
@weight(0.05)
@dataclass
class Log(Operation):
    value: Scalar
    def evaluate(self, X_np): return np.log(np.where(self.value.evaluate(X_np) <= 0, 1e-6, self.value.evaluate(X_np)))
    def __str__(self): return f"log({self.value})"

# --- CONTROL FLOW ---
@weight(0.25)
@dataclass
class IfThenElse(Main):
    condition: Condition
    then_case: Operation # Enforces operations only, as requested
    else_case: Operation

    def evaluate(self, X_np):
        mask = self.condition.evaluate(X_np)
        return np.where(mask, self.then_case.evaluate(X_np), self.else_case.evaluate(X_np))

    def __str__(self):
        return f"If({self.condition}, {self.then_case}, {self.else_case})"

# Final extraction
# Note: We removed ScalarVar and added NumericalVar
grammar = extract_grammar([Add, Subtract, Multiply, Divide, Sqrt, Log, NumericalVar, IfThenElse, CategoricalCondition, Operation], Main)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Main,Productions={
Main -> Scalar()|
	IfThenElse(condition: Condition, then_case: Operation, else_case: Operation)<0.20>

Operation -> Subtract(right: Scalar, left: Scalar)<0.17>|
	Multiply(right: Scalar, left: Scalar)<0.17>|
	Divide(right: Scalar, left: Scalar)<0.17>|
	Sqrt(value: Scalar)<0.17>|
	Log(value: Scalar)<0.17>|
	Add(right: Scalar, left: Scalar)<0.17>

Scalar -> NumericalVar(index: Annotated[int])<0.45>|
	Operation()<0.55>

Condition -> CategoricalCondition(index: Annotated[int])<1.00>
}
